# Phase 3 — Few-shot style training (variable short context, transformer)

Each batch item uses only **k** past frames (random **k** in `[min_context, max_context]`) to predict the future. The source tensor is **left-padded** to `max_context` so the architecture stays fixed; **padding is ignored** via `src_key_padding_mask` on `MotionTransformer`.

This mimics a tracker that must extrapolate after only a handful of good associations. Motion features are computed on a **single contiguous** `[context | future]` window so velocity at the first predicted frame uses the last observed box.

Data: `FewShotPaddedMotionDataset` in `few_shot_motion_dataset.py` (also exposed as `dataset.few_shot_padded_from_roots`). Increase `samples_per_window` to generate more examples per trajectory stride.

In [ ]:
import os
import torch
from torch import optim
from torch.utils.data import DataLoader

from few_shot_motion_dataset import FewShotPaddedMotionDataset, build_src_key_padding_mask
from transformer_encoder import MotionTransformer
from loss import LossFunction

MAX_CONTEXT = 24
MIN_CONTEXT = 3
SEQ_OUT_LEN = 20
STEPS = 4
SAMPLES_PER_WINDOW = 2
BATCH_SIZE = 256
NOISE_COEFF = 0.12
NOISE_PROB = 0.15
BASE_DIR = os.environ.get("MOT_DATASET_ROOT", "../../Datasets/")

train_ds = FewShotPaddedMotionDataset.from_roots(
    [f"{BASE_DIR}MOT17/train"],
    max_context=MAX_CONTEXT,
    min_context=MIN_CONTEXT,
    seq_out_len=SEQ_OUT_LEN,
    steps=STEPS,
    random_jump=False,
    noise_prob=NOISE_PROB,
    noise_coeff=NOISE_COEFF,
    random_drop_prob=0.05,
    samples_per_window=SAMPLES_PER_WINDOW,
    seed=42,
    return_context_len=True,
)
val_ds = FewShotPaddedMotionDataset.from_roots(
    [f"{BASE_DIR}MOT17/val"],
    max_context=MAX_CONTEXT,
    min_context=MIN_CONTEXT,
    seq_out_len=SEQ_OUT_LEN,
    steps=STEPS,
    noise_prob=NOISE_PROB,
    noise_coeff=NOISE_COEFF,
    random_drop_prob=0.0,
    samples_per_window=1,
    seed=43,
    return_context_len=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("train", len(train_ds), "val", len(val_ds))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 3e-4
EPOCHS = 20

model = MotionTransformer(
    input_dim=13,
    output_dim=5,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_ff=1024,
    dropout=0.1,
).to(DEVICE)
crit = LossFunction(loss1_coeff=1.0, loss2_coeff=0.35, loss3_coeff=0.25, loss4_coeff=0.0)
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS, 1))


def train_epoch():
    model.train()
    s = 0.0
    for batch in train_loader:
        src, trg, gt_src, gt_trg, k = batch
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        gt_trg = gt_trg.to(DEVICE)
        pad = build_src_key_padding_mask(k, MAX_CONTEXT, DEVICE)
        opt.zero_grad()
        out = model(src, trg[:, :-1], src_key_padding_mask=pad)
        loss = crit(out, gt_trg[:, 1:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        s += loss.item()
    return s / max(len(train_loader), 1)


@torch.no_grad()
def val_epoch():
    model.eval()
    s = 0.0
    for batch in val_loader:
        src, trg, gt_src, gt_trg, k = batch
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        gt_trg = gt_trg.to(DEVICE)
        pad = build_src_key_padding_mask(k, MAX_CONTEXT, DEVICE)
        out = model(src, trg[:, :-1], src_key_padding_mask=pad)
        s += crit(out, gt_trg[:, 1:]).item()
    return s / max(len(val_loader), 1)


print("params", sum(p.numel() for p in model.parameters()))

In [ ]:
best = float("inf")
os.makedirs("pretrained", exist_ok=True)
path = "pretrained/transformer_few_shot_varied_ctx.pth"
for ep in range(1, EPOCHS + 1):
    tr = train_epoch()
    va = val_epoch()
    sched.step()
    if va < best:
        best = va
        model.save_weight(path)
    print(f"ep {ep:02d} train {tr:.5f} val {va:.5f}")
print("best", best, path)